# Analytical Placement v3 — expanded baseline + topology springs

**Hypothesis:** the human-expert baseline encodes excellent topology and
geometry, but the tight 43% utilization leaves the optimizer no room to
improve. Strategy:

1. **Expand the baseline by 1.5×** around the canvas center — gives macros
   breathing room without changing relative arrangement.
2. **Add topology springs** — each macro is connected to its k nearest
   neighbors (in the original placement) by a spring whose rest length is
   the *original* distance. Springs preserve the expert structure.
3. **Learnable spring stiffness** — each spring has a stiffness parameter
   the optimizer can perturb, so it can selectively relax springs that
   are blocking progress.
4. **Decaying global spring weight** — early on we trust the expert
   structure (high `w_spring`); late we let wirelength + density take
   over (low `w_spring`).
5. **Resume support** — checkpoints save full state so you can add more
   iterations without restarting from scratch.

Loss = smooth HPWL  +  ReLU overlap  +  canvas hinge  +  **spring**
                                                       +  **density** (optional)

## 1. Setup

In [ ]:
import os, sys, math
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "macro_place").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

ANALYTICAL_DIR = REPO_ROOT / "submissions" / "analytical"
CHECKPOINT_DIR = ANALYTICAL_DIR / "checkpoints_v3"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import torch.nn.functional as F

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = select_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {DEVICE}")

## 2. Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display

from macro_place.loader import load_benchmark_from_dir
from submissions.analytical import losses as L
from submissions.core.eval import evaluate as tilos_evaluate

## 3. Load benchmark

In [ ]:
BENCHMARK_NAME = "ibm01"
BENCHMARK_DIR = REPO_ROOT / "external" / "MacroPlacement" / "Testcases" / "ICCAD04" / BENCHMARK_NAME

benchmark, plc = load_benchmark_from_dir(str(BENCHMARK_DIR))
print(benchmark)

## 4. Expand baseline by 1.5×

Scale all movable macros' positions 1.5× outward from the canvas center,
then clamp to the canvas. This preserves *relative* layout but gives each
macro more room to move. Fixed macros stay where the benchmark placed them.

In [ ]:
def expand_baseline(benchmark, scale=1.5):
    """Scale movable hard-macro positions by `scale` around canvas center."""
    num_hard = benchmark.num_hard_macros
    positions = benchmark.macro_positions.clone().float()
    sizes     = benchmark.macro_sizes.float()
    fixed     = benchmark.macro_fixed[:num_hard]

    center = torch.tensor(
        [benchmark.canvas_width / 2.0, benchmark.canvas_height / 2.0],
        dtype=positions.dtype,
    )

    expanded = center + scale * (positions[:num_hard] - center)
    movable  = ~fixed
    # Apply only to movable rows.
    new_hard = positions[:num_hard].clone()
    new_hard[movable] = expanded[movable]
    positions[:num_hard] = new_hard

    # Clamp to canvas so nothing escapes.
    half = sizes[:num_hard] / 2.0
    positions[:num_hard, 0] = positions[:num_hard, 0].clamp(
        half[:, 0], float(benchmark.canvas_width)  - half[:, 0]
    )
    positions[:num_hard, 1] = positions[:num_hard, 1].clamp(
        half[:, 1], float(benchmark.canvas_height) - half[:, 1]
    )
    return positions


init_positions = expand_baseline(benchmark, scale=1.5)
print(f"Expanded {benchmark.num_hard_macros} macros around canvas center.")

## 5. Build topology springs (top-k nearest neighbors)

For each hard macro, find its k closest neighbors **in the original
baseline placement**. Each (i, j) pair becomes a spring with rest length
= original baseline distance. Edges are deduplicated.

Stiffness is initialized so that `softplus(stiffness) = 1.0` for every
spring, and is registered as a learnable parameter so the optimizer can
perturb it.

In [ ]:
def build_springs(benchmark, k=8):
    """
    Returns (spring_i, spring_j, rest_length).

    spring_i, spring_j: long tensors [E] — endpoints of each undirected spring.
    rest_length:         float tensor [E] — distance in the original baseline.
    """
    num_hard = benchmark.num_hard_macros
    pos = benchmark.macro_positions[:num_hard].float()

    diff = pos.unsqueeze(0) - pos.unsqueeze(1)         # [N, N, 2]
    dist = (diff * diff).sum(-1).sqrt()                # [N, N]
    dist = dist + torch.eye(num_hard) * 1.0e9          # ignore self

    topk_idx = dist.topk(k, dim=-1, largest=False).indices    # [N, k]

    # Deduplicate edges by canonical ordering (i < j).
    edges = set()
    for i in range(num_hard):
        for jj in range(k):
            j = int(topk_idx[i, jj])
            edges.add((min(i, j), max(i, j)))
    edges = sorted(edges)

    spring_i = torch.tensor([e[0] for e in edges], dtype=torch.long)
    spring_j = torch.tensor([e[1] for e in edges], dtype=torch.long)
    rest_length = (pos[spring_i] - pos[spring_j]).norm(dim=-1)
    return spring_i, spring_j, rest_length


K_NEIGHBORS = 8
spring_i, spring_j, rest_length = build_springs(benchmark, k=K_NEIGHBORS)
print(f"Built {len(spring_i)} springs from top-{K_NEIGHBORS} neighbors.")
print(f"  Rest length:  min={float(rest_length.min()):.3f}  "
      f"max={float(rest_length.max()):.3f}  "
      f"mean={float(rest_length.mean()):.3f} μm")

## 6. Spring loss function

Hooke's-law style: `loss = Σ softplus(stiffness_e) · (||p_i − p_j|| − rest_e)²`.
Softplus keeps stiffness positive while still being smooth and learnable.

In [ ]:
def spring_loss(positions, spring_i, spring_j, rest_length, stiffness_param):
    """Differentiable spring loss with learnable per-edge stiffness."""
    delta = positions[spring_i] - positions[spring_j]            # [E, 2]
    current_dist = (delta * delta).sum(-1).clamp(min=1e-12).sqrt()
    extension = current_dist - rest_length
    effective_stiffness = F.softplus(stiffness_param)
    return (effective_stiffness * extension * extension).sum()

## 7. Visualization helpers

In [ ]:
def plot_placement(ax, positions, benchmark, title="", show_springs=False,
                   spring_i=None, spring_j=None, max_springs=400):
    ax.clear()
    ax.set_xlim(0, benchmark.canvas_width)
    ax.set_ylim(0, benchmark.canvas_height)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")

    pos = positions.detach().cpu().numpy()

    # Springs underneath (light red lines).
    if show_springs and spring_i is not None:
        n = len(spring_i)
        idx = np.arange(n) if n <= max_springs else np.random.choice(n, max_springs, replace=False)
        si = spring_i.cpu().numpy()
        sj = spring_j.cpu().numpy()
        for k in idx:
            i, j = int(si[k]), int(sj[k])
            ax.plot(
                [pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                color="crimson", alpha=0.15, linewidth=0.3, zorder=1,
            )

    sizes = benchmark.macro_sizes.cpu().numpy()
    fixed = benchmark.macro_fixed.cpu().numpy()
    for i in range(benchmark.num_hard_macros):
        x = pos[i, 0] - sizes[i, 0] / 2.0
        y = pos[i, 1] - sizes[i, 1] / 2.0
        color = "lightgray" if fixed[i] else "steelblue"
        ax.add_patch(mpatches.Rectangle(
            (x, y), sizes[i, 0], sizes[i, 1],
            facecolor=color, edgecolor="black",
            alpha=0.55, linewidth=0.4, zorder=2,
        ))


def plot_loss(ax, history):
    ax.clear()
    if not history["step"]:
        return
    for key, label in [("wl", "smooth WL"), ("overlap", "overlap"),
                       ("spring", "spring"), ("density", "density"),
                       ("total", "total")]:
        if key in history and any(v > 0 for v in history[key]):
            lw = 2 if key == "total" else 1
            ax.plot(history["step"], history[key], label=label, linewidth=lw)
    ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss (log)")
    ax.set_title("Loss components")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")


# Compare baseline vs expanded with springs visible.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_placement(axes[0], benchmark.macro_positions, benchmark,
               title=f"{benchmark.name} — baseline (1.0×)")
plot_placement(axes[1], init_positions, benchmark,
               title=f"{benchmark.name} — expanded 1.5× + springs (faint red)",
               show_springs=True, spring_i=spring_i, spring_j=spring_j)
plt.tight_layout(); plt.show()

## 8. Training function with springs + resume support

Pass `resume_from=<checkpoint path>` to continue from a saved state
(positions, stiffness, optimizer, scheduler, history all restored).

In [ ]:
def train_v3(
    benchmark,
    plc,
    init_positions,
    spring_i, spring_j, rest_length,
    *,
    n_epochs=100,
    steps_per_epoch=100,
    lr=3e-3,
    lr_stiffness=1e-2,
    gamma_start=0.01,
    gamma_end=0.002,
    w_overlap=80.0,
    w_canvas=200.0,
    w_spring_start=200.0,    # high early: trust baseline topology
    w_spring_end=10.0,       # low late:  let WL + density take over
    w_density=0.0,           # set >0 to add density spreading
    n_bins=8,
    checkpoint_dir=CHECKPOINT_DIR,
    device=DEVICE,
    seed=42,
    resume_from=None,        # path to a .pt checkpoint
):
    """Training loop. Supports clean start or resume from checkpoint."""
    torch.manual_seed(seed)
    total_steps = n_epochs * steps_per_epoch

    # ── State: either fresh or restored ───────────────────────────────────
    if resume_from is not None:
        ckpt = torch.load(resume_from, map_location="cpu", weights_only=False)
        positions = ckpt["positions"].to(device).clone().requires_grad_(True)
        stiffness = ckpt["stiffness"].to(device).clone().requires_grad_(True)
        history   = ckpt["history"]
        start_step = ckpt["step"]
        print(f"Resuming from {resume_from}")
        print(f"  starting at step {start_step}, epoch {start_step // steps_per_epoch}")
    else:
        positions = init_positions.clone().to(device).requires_grad_(True)
        # One stiffness param per spring, initialized so softplus(s) ≈ 1.
        stiffness = torch.full(
            (len(spring_i),), math.log(math.e - 1.0),
            device=device, dtype=torch.float32,
        ).requires_grad_(True)
        history = {"step": [], "wl": [], "overlap": [], "spring": [],
                   "density": [], "canvas": [], "total": [], "w_spring": []}
        start_step = 0

    sizes_dev   = benchmark.macro_sizes.to(device, dtype=torch.float32)
    fixed_dev   = benchmark.macro_fixed.to(device)
    anchor      = benchmark.macro_positions.to(device, dtype=torch.float32)
    spring_i_d  = spring_i.to(device)
    spring_j_d  = spring_j.to(device)
    rest_len_d  = rest_length.to(device)

    padded_nets, mask, net_weights = L.prepare_net_tensors(benchmark, device)
    n_pos = positions.shape[0]
    out_of_bounds = padded_nets >= n_pos
    padded_nets = padded_nets.clamp(max=n_pos - 1)
    mask = mask & ~out_of_bounds

    # Two parameter groups: positions (main LR) and stiffness (slower LR).
    optimizer = torch.optim.Adam([
        {"params": [positions], "lr": lr},
        {"params": [stiffness], "lr": lr_stiffness},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=lr / 100
    )

    if resume_from is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])

    best_total = float("inf")
    best_positions = positions.detach().cpu().clone()
    epoch_log = []

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig_handle = ipy_display(fig, display_id=True)
    t0 = time.perf_counter()

    def lerp(s, e, frac):
        return s + (e - s) * max(0.0, min(1.0, frac))

    for step in range(start_step + 1, start_step + total_steps + 1):
        frac = (step - start_step) / total_steps
        gamma    = lerp(gamma_start,   gamma_end,   frac)
        w_spring = lerp(w_spring_start, w_spring_end, frac)

        optimizer.zero_grad()

        wl_term      = L.smooth_hpwl(positions, padded_nets, mask, net_weights, gamma=gamma)
        overlap_term = L.overlap_penalty(positions, sizes_dev, benchmark.num_hard_macros)
        canvas_term  = L.canvas_penalty(
            positions, sizes_dev, benchmark.canvas_width, benchmark.canvas_height
        )
        spring_term  = spring_loss(positions, spring_i_d, spring_j_d, rest_len_d, stiffness)

        if w_density > 0:
            density_term = L.density_penalty(
                positions, sizes_dev, benchmark.num_hard_macros,
                benchmark.canvas_width, benchmark.canvas_height, n_bins=n_bins,
            )
        else:
            density_term = torch.tensor(0.0, device=device)

        # Tiny L2 regularization on stiffness keeps it near init unless
        # the optimizer has a strong reason to perturb it.
        stiffness_reg = 1e-4 * (stiffness ** 2).sum()

        loss = (
            wl_term
            + w_overlap * overlap_term
            + w_canvas  * canvas_term
            + w_spring  * spring_term
            + w_density * density_term
            + stiffness_reg
        )
        loss.backward()

        with torch.no_grad():
            positions.grad[fixed_dev] = 0.0
        optimizer.step()
        scheduler.step()
        with torch.no_grad():
            positions.data[fixed_dev] = anchor[fixed_dev]

        loss_f = float(loss.detach())
        if loss_f < best_total:
            best_total = loss_f
            best_positions = positions.detach().cpu().clone()

        history["step"].append(step)
        history["wl"].append(float(wl_term.detach()))
        history["overlap"].append(max(float(overlap_term.detach()), 1e-12))
        history["spring"].append(max(float(spring_term.detach()), 1e-12))
        history["density"].append(max(float(density_term.detach()), 1e-12))
        history["canvas"].append(max(float(canvas_term.detach()), 1e-12))
        history["total"].append(loss_f)
        history["w_spring"].append(w_spring)

        if step % steps_per_epoch == 0:
            epoch_idx = step // steps_per_epoch
            elapsed = time.perf_counter() - t0

            plot_placement(
                axes[0], positions, benchmark,
                title=(
                    f"{benchmark.name}  epoch {epoch_idx}  "
                    f"γ={gamma:.4f}  w_spring={w_spring:.1f}"
                ),
                show_springs=True,
                spring_i=spring_i, spring_j=spring_j,
            )
            plot_loss(axes[1], history)
            fig.tight_layout()
            fig_handle.update(fig)

            stiff = F.softplus(stiffness).detach()
            print(
                f"epoch {epoch_idx:>3d}  "
                f"total={loss_f:.2f}  wl={float(wl_term.detach()):.2f}  "
                f"spring={float(spring_term.detach()):.2f}  "
                f"overlap={float(overlap_term.detach()):.4f}  "
                f"stiff[mean]={float(stiff.mean()):.2f}±{float(stiff.std()):.2f}  "
                f"elapsed={elapsed:.1f}s"
            )

            ckpt_path = checkpoint_dir / f"{benchmark.name}_v3_epoch{epoch_idx:03d}.pt"
            torch.save({
                "benchmark": benchmark.name,
                "epoch": epoch_idx,
                "step":  step,
                "positions":      positions.detach().cpu(),
                "best_positions": best_positions,
                "stiffness":      stiffness.detach().cpu(),
                "spring_i":       spring_i,
                "spring_j":       spring_j,
                "rest_length":    rest_length,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "history": history,
                "config":  {
                    "lr": lr, "lr_stiffness": lr_stiffness,
                    "w_overlap": w_overlap, "w_canvas": w_canvas,
                    "w_spring_start": w_spring_start, "w_spring_end": w_spring_end,
                    "w_density": w_density, "n_bins": n_bins, "seed": seed,
                },
            }, ckpt_path)
            epoch_log.append({
                "epoch": epoch_idx, "step": step,
                "path":  str(ckpt_path),
                "total": loss_f, "wl": float(wl_term.detach()),
                "spring": float(spring_term.detach()),
            })

    plt.close(fig)
    return best_positions, stiffness.detach().cpu(), history, epoch_log

## 9. Run training — 100 epochs

In [ ]:
positions, stiffness, history, epoch_log = train_v3(
    benchmark, plc, init_positions,
    spring_i, spring_j, rest_length,
    n_epochs=100,
    steps_per_epoch=100,
    w_spring_start=200.0,
    w_spring_end=10.0,
)

print("\nLast 5 epochs:")
for row in epoch_log[-5:]:
    print(f"  epoch {row['epoch']:>3d}  total={row['total']:.2f}  "
          f"wl={row['wl']:.2f}  spring={row['spring']:.2f}")

## 10. Resume from checkpoint — add more iterations

Edit `RESUME_PATH` to point at any saved checkpoint and run this cell.
Pick up exactly where the previous run left off (positions, stiffness,
optimizer momentum, LR schedule, full history). `n_epochs` here is the
number of *additional* epochs to run.

In [ ]:
# Find the most recent checkpoint automatically (or hardcode a path).
all_ckpts = sorted(CHECKPOINT_DIR.glob(f"{BENCHMARK_NAME}_v3_epoch*.pt"))
RESUME_PATH = all_ckpts[-1] if all_ckpts else None
print(f"Resuming from: {RESUME_PATH}")

if RESUME_PATH is not None:
    positions, stiffness, history, epoch_log = train_v3(
        benchmark, plc, init_positions,
        spring_i, spring_j, rest_length,
        n_epochs=50,                   # additional epochs
        steps_per_epoch=100,
        w_spring_start=10.0,           # springs already weak, keep weak
        w_spring_end=2.0,
        resume_from=str(RESUME_PATH),
    )
    print("\nLast 5 epochs:")
    for row in epoch_log[-5:]:
        print(f"  epoch {row['epoch']:>3d}  total={row['total']:.2f}")

## 11. Legalize and evaluate with TILOS

In [ ]:
from submissions.analytical.placer import AnalyticalPlacer
from submissions.core.eval import visualize

placer = AnalyticalPlacer(n_steps=0, legalize_iters=600, device=DEVICE)
pos = positions.clone().to(DEVICE).requires_grad_(True)
sizes_dev = benchmark.macro_sizes.to(DEVICE, dtype=torch.float32)
fixed_dev = benchmark.macro_fixed.to(DEVICE)
pos = placer._clamp_to_canvas(pos, sizes_dev, benchmark)
pos = placer._restore_fixed(pos, benchmark, fixed_dev)
pos = placer._legalize_push_apart(pos, sizes_dev, benchmark, fixed_dev)
pos = placer._restore_fixed(pos, benchmark, fixed_dev).detach().cpu()

result = tilos_evaluate(pos, benchmark, plc)
print(
    f"Proxy: {result['proxy_cost']:.4f}  "
    f"WL: {result['wirelength_cost']:.4f}  "
    f"Density: {result['density_cost']:.4f}  "
    f"Congestion: {result['congestion_cost']:.4f}  "
    f"Overlaps: {result['overlap_count']}"
)

final_path = CHECKPOINT_DIR / f"{benchmark.name}_v3_final.pt"
torch.save({"positions": pos, "tilos": result, "stiffness": stiffness}, final_path)
visualize(pos, benchmark, plc=plc, save_path=str(CHECKPOINT_DIR / f"{benchmark.name}_v3_final.png"))
print(f"Saved -> {final_path}")

## 12. Quick-test ideas to try next

Below are runnable variants of the spring idea. Each is a small
swap-in change you can try without rewriting the training loop.

**Idea A — Net-aware springs.**  Use shared-net connectivity instead of
spatial proximity. Macros connected by short, low-fanout nets get
stronger springs. Aligns spring topology with the actual wirelength
objective.

**Idea B — Springs + density combined.**  Add a density spreading term
to the spring loss. Springs preserve topology; density prevents
wirelength collapse. Best of both worlds.

**Idea C — Annealed expansion.**  Expansion scale starts at 1.5×, anneals
back to 1.0× over training. Macros explore a wider region early, then
settle into a tighter configuration.

**Idea D — Per-macro mobility.**  Give each macro a learnable temperature.
Frequently-connected macros become "hotter" (move more); peripheral
macros cool down (move less). Equivalent to per-macro learning rate.

**Idea E — Gradient noise (Langevin).**  Add `σ · N(0,1)` to the gradient
before each step, with `σ` decaying over time. Escapes local minima
early, then becomes deterministic at the end.

Below: Idea A and Idea B implemented as quick cells.

### Idea A — Net-aware springs

In [ ]:
def build_net_springs(benchmark, max_net_size=12):
    """
    Build springs from net connectivity.

    For each net with <= max_net_size hard-macro pins, add a spring
    between every pair of hard macros in that net. Rest length is the
    distance between them in the baseline.
    """
    num_hard = benchmark.num_hard_macros
    pos = benchmark.macro_positions[:num_hard].float()
    edges = set()

    for net in benchmark.net_nodes:
        hard = [int(n) for n in net.tolist() if 0 <= int(n) < num_hard]
        if not (2 <= len(hard) <= max_net_size):
            continue
        hard = sorted(set(hard))
        for a in range(len(hard)):
            for b in range(a + 1, len(hard)):
                edges.add((hard[a], hard[b]))

    edges = sorted(edges)
    si = torch.tensor([e[0] for e in edges], dtype=torch.long)
    sj = torch.tensor([e[1] for e in edges], dtype=torch.long)
    rl = (pos[si] - pos[sj]).norm(dim=-1)
    return si, sj, rl


# To run: replace the spring tensors and re-train.
# net_spring_i, net_spring_j, net_rest_length = build_net_springs(benchmark, max_net_size=12)
# print(f"Net-aware springs: {len(net_spring_i)} edges")
# positions_A, _, _, _ = train_v3(
#     benchmark, plc, init_positions,
#     net_spring_i, net_spring_j, net_rest_length,
#     n_epochs=100, steps_per_epoch=100,
#     w_spring_start=100.0, w_spring_end=5.0,
# )

### Idea B — Springs + density combined

In [ ]:
# Just set w_density > 0 when calling train_v3 — both losses active.
#
# positions_B, _, _, _ = train_v3(
#     benchmark, plc, init_positions,
#     spring_i, spring_j, rest_length,
#     n_epochs=100, steps_per_epoch=100,
#     w_spring_start=150.0, w_spring_end=15.0,
#     w_density=1000.0,        # ← active density penalty
#     n_bins=8,
# )